# 13 — Stereo vision: depth, epipolar geometry, $F$/$E$, triangulation

**Needed for the assignment:** no (course / exam topic). **Study:** 3 h, **Practice:** 2 h.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

## Idea

One image loses depth, but two images of the same point give it back by **triangulation**. In **standard stereo** (parallel cameras, same rows) depth is just $Z=bf/d$. In general, a point in image 1 must lie on a **line** in image 2 (the epipolar constraint). That line comes from the **essential** matrix $E$ (calibrated cameras) or the **fundamental** matrix $F$ (uncalibrated), which is estimated from at least 8 correspondences (normalized 8-point method).

## Key formulas

## Stereo Vision and Epipolar Geometry

### 1. Standard stereo reconstruction

For rectified stereo cameras, corresponding points have the same vertical coordinate.

The disparity is:

$$
d=u_1-u_2
$$

The 3D coordinates in the first camera's coordinate system are:

$$
Z=\frac{bf}{d}
$$

$$
X=-\frac{b(u_1+u_2)}{2d}
$$

$$
Y=\frac{bv_1}{d}
$$

Where:
- \(b\): baseline (distance between the cameras)
- \(f\): focal length
- \(d\): disparity
- \((u_1,v_1)\), \((u_2,v_2)\): corresponding image points

**Key idea:** Larger disparity means smaller depth (the object is closer).

### 2. Essential matrix

The essential matrix describes the geometric relationship between two cameras in calibrated coordinates.

$$
\mathbf{u}_2^T E\mathbf{u}_1=0
$$

$$
E=[\mathbf{t}]_\times R
$$

Properties:
- 5 degrees of freedom (3 for rotation and 2 for translation direction).
- Rank 2.
- Two nonzero singular values that are equal.

The skew-symmetric matrix of translation is:

$$
[\mathbf{t}]_\times=
\begin{bmatrix}
0&-t_3&t_2\\
t_3&0&-t_1\\
-t_2&t_1&0
\end{bmatrix}
$$

### 3. Fundamental matrix

The fundamental matrix describes the relationship between two image points in pixel coordinates.

$$
\mathbf{u}_2^T F\mathbf{u}_1=0
$$

Its relationship with the essential matrix is:

$$
F=K_2^{-T}EK_1^{-1}
$$

Where \(K_1\) and \(K_2\) are the camera intrinsic matrices.

Properties:
- 7 degrees of freedom.
- Rank 2.

### 4. Epipolar lines and epipoles

Given a point in one image, its corresponding epipolar line in the other image is:

$$
\mathbf{l}_2=F\mathbf{u}_1
$$

$$
\mathbf{l}_1=F^T\mathbf{u}_2
$$

The epipoles are the null-space vectors of the fundamental matrix:

$$
F\mathbf{e}_1=\mathbf{0}
$$

$$
F^T\mathbf{e}_2=\mathbf{0}
$$

The corresponding point must lie on its epipolar line.

### 5. Eight-point algorithm

For each corresponding point pair \((u_1,v_1)\) and \((u_2,v_2)\), construct one row:

$$
[u_2u_1,\ u_2v_1,\ u_2,\ 
v_2u_1,\ v_2v_1,\ v_2,\ 
u_1,\ v_1,\ 1]
$$

Stack at least 8 rows to form \(A\), then solve:

$$
A\mathbf{f}=\mathbf{0}
$$

The solution is the right singular vector corresponding to the smallest singular value of \(A\). Reshape \(\mathbf{f}\) into a \(3\times3\) matrix \(F\).

### 6. Enforcing rank 2 and denormalization

The estimated fundamental matrix may not have rank 2 due to noise.

Use SVD:

$$
F=U\Sigma V^T
$$

Set the smallest singular value to zero:

$$
F_{\text{rank-2}}=
U
\begin{bmatrix}
\delta_1&0&0\\
0&\delta_2&0\\
0&0&0
\end{bmatrix}
V^T
$$

If points were normalized before estimation, denormalize:

$$
F=T_2^T F_n T_1
$$

### 7. Triangulation

Triangulation estimates a 3D point from its corresponding image points and the two camera projection matrices.

For a camera projection matrix \(P\), with rows \(\mathbf{p}_1^T,\mathbf{p}_2^T,\mathbf{p}_3^T\), construct two equations:

$$
u\mathbf{p}_3^T\mathbf{X}
-\mathbf{p}_1^T\mathbf{X}=0
$$

$$
v\mathbf{p}_3^T\mathbf{X}
-\mathbf{p}_2^T\mathbf{X}=0
$$

Each camera contributes two rows to \(A\). With two cameras, stack four rows and solve:

$$
A\mathbf{X}=\mathbf{0}
$$

Use SVD and take the right singular vector corresponding to the smallest singular value. Convert the resulting homogeneous 3D point to Cartesian coordinates by dividing by its fourth coordinate.

### 8. Recovering camera pose from the essential matrix

Decomposing \(E\) using SVD gives four possible combinations of rotation \(R\) and translation direction \(\mathbf{t}\).

To select the correct solution, triangulate the points for each candidate and check **cheirality**: the reconstructed points must be in front of both cameras (positive depth in both camera coordinate systems).

# Worked example 1 (on paper): depth and its precision

Lab 1 numbers: $f=3740$ px, $b=160$ mm.

-   $d=200$: $Z=\frac{3740\cdot160}{200}=2992$ mm, and $d=199$ gives $3007$ mm. One pixel of disparity error costs about $15$ mm.

-   $d=50$: $Z=11968$ mm, and $d=49$ gives $12212$ mm. Now one pixel costs about $244$ mm.

Far points have small $d$, so their relative depth error is large ($|\Delta Z|/Z\approx|\Delta d|/d$). A wider baseline gives a larger $d$ and better precision.

# Worked example 2 (on paper): $E$ for a sideways move

$R=I$, $\mathbf{t}=(1,0,0)$:  $E=[\mathbf{t}]_\times=\begin{bmatrix}0&0&0\\0&0&-1\\0&1&0\end{bmatrix}$.\
For $\mathbf{u}_1=(x_1,y_1,1)$: $E\mathbf{u}_1=(0,\,-1,\,y_1)$, so $\mathbf{u}_2^TE\mathbf{u}_1=-y_2+y_1=0\Rightarrow y_1=y_2$. The epipolar lines are the image rows: this is exactly standard stereo (Lab 1 searches along one row).

# The same in Python / NumPy

In [2]:
import numpy as np

# --- 1) standard stereo depth (Lab 1 numbers: f = 3740 px, b = 160 mm) ---
f, b = 3740.0, 160.0
for d in (200, 199, 50, 49):
    print(f"d = {d:3d} px  ->  Z = {f * b / d:8.1f} mm")

# --- 2) essential matrix of a pure sideways move ---
def skew(t):
    return np.array([[0, -t[2], t[1]], [t[2], 0, -t[0]], [-t[1], t[0], 0]])
E = skew([1.0, 0, 0]) @ np.eye(3)
print("E =\n", E)
u1, u2 = np.array([0.3, 0.2, 1]), np.array([0.1, 0.2, 1])
print("u2^T E u1 =", u2 @ E @ u1, "(same row -> 0)")

# --- 3) normalized 8-point method on synthetic data ---
rng = np.random.default_rng(0)
K = np.array([[700, 0, 320], [0, 700, 240], [0, 0, 1.0]])
a = np.radians(10)
R = np.array([[np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]])
t = np.array([1.0, 0.1, 0.2])                       # camera 2: x2 = R x1 + t
X = rng.uniform([-2, -2, 6], [2, 2, 10], size=(40, 3))

def project(K, Xc):
    x = Xc @ K.T
    return x[:, :2] / x[:, 2:3]
p1 = project(K, X) + rng.normal(0, 0.3, (40, 2))
p2 = project(K, X @ R.T + t) + rng.normal(0, 0.3, (40, 2))

def norm_T(p):
    m = p.mean(axis=0)
    s = np.sqrt(2) / np.mean(np.linalg.norm(p - m, axis=1))
    return np.array([[s, 0, -s * m[0]], [0, s, -s * m[1]], [0, 0, 1]])
def hom(p):
    return np.column_stack([p, np.ones(len(p))])

T1, T2 = norm_T(p1), norm_T(p2)
q1, q2 = hom(p1) @ T1.T, hom(p2) @ T2.T
A = np.array([[x2 * x1, x2 * y1, x2, y2 * x1, y2 * y1, y2, x1, y1, 1]
              for (x1, y1, _), (x2, y2, _) in zip(q1, q2)])
_, _, Vt = np.linalg.svd(A)
F0 = Vt[-1].reshape(3, 3)
U, S, Vt2 = np.linalg.svd(F0)
F0 = U @ np.diag([S[0], S[1], 0]) @ Vt2               # force rank 2
F = T2.T @ F0 @ T1                                    # denormalize
F /= np.linalg.norm(F)
print("singular values of F:", np.round(np.linalg.svd(F)[1], 6))
res = np.abs(np.sum((hom(p2) @ F) * hom(p1), axis=1))
print("mean |u2^T F u1| =", f"{res.mean():.2e}")

# --- 4) linear triangulation of the first point ---
P1 = K @ np.hstack([np.eye(3), np.zeros((3, 1))])
P2 = K @ np.hstack([R, t.reshape(3, 1)])
(u, v), (u_, v_) = p1[0], p2[0]
A = np.array([u * P1[2] - P1[0], v * P1[2] - P1[1], u_ * P2[2] - P2[0], v_ * P2[2] - P2[1]])
_, _, Vt = np.linalg.svd(A)
Xh = Vt[-1]
print("triangulated:", np.round(Xh[:3] / Xh[3], 3), " true:", np.round(X[0], 3))

d = 200 px  ->  Z =   2992.0 mm
d = 199 px  ->  Z =   3007.0 mm
d =  50 px  ->  Z =  11968.0 mm
d =  49 px  ->  Z =  12212.2 mm
E =
 [[ 0.  0.  0.]
 [ 0.  0. -1.]
 [ 0.  1.  0.]]
u2^T E u1 = 0.0 (same row -> 0)
singular values of F: [0.9999 0.0119 0.    ]
mean |u2^T F u1| = 3.17e-02
triangulated: [ 0.552 -0.922  6.189]  true: [ 0.548 -0.921  6.164]


## Course / lab link

Lab 1: block matching with SAD along the row (standard stereo, Middlebury "Art"), then $Z=fB/d$, then a 3D plot. lec02 also covers rectification (warping both images so that epipolar lines become rows) and planar motion (a car on a road: only 2 angles, so 2 correspondences are enough).

## Practice

Write your solution in the empty cell under each exercise. The **expected result** tells you what you should get, not how to get it. Grey *provided data* cells are already filled in; run them first.

### Exercise 1

With $f=700$ px and $b=0.12$ m, what disparity does a point at 3 m have? How large is the depth error for $\pm1$ px?

> **Expected result:** $d=28$ px. $d=27$ gives $3.111$ m and $d=29$ gives $2.897$ m, so about $\pm0.11$ m.

In [3]:
# Exercise 1: your solution


### Exercise 2

Compute $E$ for $R=I$, $\mathbf t=(0,1,0)$. What does the epipolar constraint say now?

> **Expected result:** $E=\begin{bmatrix}0&0&1\\0&0&0\\-1&0&0\end{bmatrix}$, so $\mathbf u_2^TE\mathbf u_1=x_2-x_1=0$: matching points are in the same **column** (vertical stereo).

In [4]:
# Exercise 2: your solution


### Exercise 3

Using the example's data (run the example cell above first), remove the normalization from the 8-point code and compare the mean distance of the points to their epipolar lines. Then remove the rank-2 step and look at the singular values of $F$.

> **Expected result:** Mean point-to-epipolar-line distance: $\approx0.35$ px with normalization, $\approx4.5$ px without. Without the rank-2 step the third singular value is $\approx7\cdot10^{-7}$: small, but not 0, so $F$ is not a valid rank-2 matrix.

In [5]:
# Exercise 3: your solution


### Exercise 4

Triangulate all 40 points of the example and report the mean 3D error.

> **Expected result:** Mean error $\approx0.036$, max $\approx0.155$ (in scene units; the points are 6–10 units away).

In [6]:
# Exercise 4: your solution
